<a href="https://colab.research.google.com/github/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA/blob/main/notebooks/00_comun/00_configuracion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Configuración del entorno

**Autora:** Michelle

**Proyecto:** ¿cómo se ha transformado la Altillanura y qué papel juegan el clima, el fuego y el suelo en la expansión agrícola?

**Propósito:** dejar listo el entorno en Colab o en local y explicar la celda de rutas que usan todos los cuadernos.

**Dónde están los datos:**
- `datos/comunes/` (en el repo): datos que usa más de una parte. Los CSV livianos van en Git; los `.gpkg` y archivos pesados, en DVC (se bajan con `dvc pull` o con la función `traer`).
- `MyDrive/proyecto-altillanura-STDG/datos/crudos/` (en Drive): descargas originales pesadas (MGN, CSV de FIRMS, copia del ONI). Solo las usan los cuadernos 01, 03 y 10.
- `modulos/clima_fuego/graficas/` y `modulos/clima_fuego/resultados/`: figuras y tablas de resultados de clima y fuego.

**Uso:** copiar la celda del paso 2 al inicio de cada cuaderno nuevo.

## Paso 1. Instalar librerías
Se ejecuta en cada sesión de Colab, porque Colab borra lo instalado al cerrarse. En local ya vienen con `pip install -r requirements.txt`.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q rasterio rioxarray rasterstats pykrige libpysal esda spreg mapclassify

## Paso 2. Celda de rutas (copiar al inicio de cada cuaderno)
- En **Colab** clona el repo en `/content/Proyecto-ALTILLANURA` y monta Drive para los crudos.
- En **local** toma como raíz la carpeta del repo (dos niveles arriba del cuaderno).
- `traer(ruta)` baja de DVC un archivo pesado cuando hace falta.

In [ ]:
# Rutas del proyecto: funcionan en Colab y en local (ver 00_configuracion)
import os, sys, subprocess
EN_COLAB = 'google.colab' in sys.modules
REPO = 'https://github.com/DanielFelipeRuizBermudez/Proyecto-ALTILLANURA.git'

if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Los datos crudos pesados (MGN, CSV de FIRMS, copia del ONI) siguen en Drive
    CRUDOS = '/content/drive/MyDrive/proyecto-altillanura-STDG/datos/crudos'
    RAIZ = '/content/Proyecto-ALTILLANURA'
    if not os.path.exists(RAIZ):
        !git clone -q {REPO} {RAIZ}
    # Mientras el PR #3 no esté unido a main, los datos comunes están en esta rama
    if not os.path.exists(f'{RAIZ}/datos/comunes/oni_mensual.csv'):
        !git -C {RAIZ} fetch -q origin feature/clima-fuego-datos
        !git -C {RAIZ} checkout -q feature/clima-fuego-datos
else:
    RAIZ = os.path.abspath(os.path.join('..', '..'))   # el cuaderno está en notebooks/<etapa>/
    CRUDOS = os.path.join(RAIZ, 'datos', 'crudos')

COMUNES = os.path.join(RAIZ, 'datos', 'comunes')
GRAFICAS = os.path.join(RAIZ, 'modulos', 'clima_fuego', 'graficas')
os.makedirs(GRAFICAS, exist_ok=True)

def traer(ruta):
    """Baja de DVC un archivo pesado del repo (por ejemplo un .gpkg) si todavía no está."""
    if os.path.exists(os.path.join(RAIZ, ruta)):
        return
    if EN_COLAB:
        from google.colab import auth
        auth.authenticate_user()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dvc[gs]'], check=True)
    subprocess.run([sys.executable, '-m', 'dvc', 'pull', '-q', ruta], cwd=RAIZ, check=True)
    print('Bajado de DVC:', ruta)

print('Raíz del repo:', RAIZ)
print('Crudos:', CRUDOS, '| existe:', os.path.exists(CRUDOS))


## Paso 3. Verificar el entorno

In [ ]:
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm

print('pandas', pd.__version__)
print('geopandas', gpd.__version__)
print('statsmodels', sm.__version__)
print('Datos comunes:', sorted(os.listdir(COMUNES)))
print('Todo listo')